### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="parkinsons_biomedical_voice_measurements",
    dataset_year="2007",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C59C74",
    download_description="""
We get the 2007 data from the UCI repository.

wget https://archive.ics.uci.edu/static/public/174/parkinsons.zip && unzip parkinsons.zip parkinsons.data && rm parkinsons.zip && mkdir -p local-data-warehouse/parkinsons_biomedical_voice_measurements && mv parkinsons.data local-data-warehouse/parkinsons_biomedical_voice_measurements/
""",
    # References
    academic_reference_bibtex="""@article{little2007exploiting,
  title={Exploiting nonlinear recurrence and fractal scaling properties for voice disorder detection},
  author={Little, Max and Mcsharry, Patrick and Roberts, Stephen and Costello, Declan and Moroz, Irene},
  journal={Nature Precedings},
  pages={1--1},
  year={2007},
  publisher={Nature Publishing Group UK London}
}
""",
    academic_reference_bibtex_key="little2007exploiting",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped", "WrongDomain"], # Maybe also temporal but not of importance for the task.
    curation_comments="""
We start with the data from UCI.

- We decode the group ID and session ID (time index per patient) from the name column.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="status",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="status",
    group_on="patient_id",
    group_time_on="session_number",
    group_labels="per_group",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "parkinsons.data")

# Split session number from patient ID, split on the last _
df[["patient_id", "session_number"]] = df["name"].str.rsplit("_", n=1, expand=True)
df["session_number"] = df["session_number"].astype(int)
df = df.drop(columns=["name"])

as_cat_type = ["status", "patient_id"]
df[as_cat_type] = df[as_cat_type].astype("category")
print("Loaded data shape:", df.shape)

df = df.sample(frac=1, random_state=42).sort_values(by=["patient_id", "session_number"]).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
    random_state=split_random_state,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="We create a default grouped split with label-per-group to judge if model can learn to predict the state of a patient given a set of recordings.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()